In [28]:
import pandas as pd
import re
import numpy as np
import soccerdata as sd
from abc import ABC, abstractmethod
from functools import wraps
import time


pd.set_option("display.max_columns", None)

seasons = ["2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]

def timeit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"Func: {func.__name__} | Czas: {time.time() - start:.3f}s")
        return result
    return wrapper

class FetchData(ABC):
    def __init__(self, season_list: list):
        self.season_list = season_list
        self.url_manager = {}

    @abstractmethod
    def get_data(self) -> pd.DataFrame:
        """Główny punkt wejścia do pobierania danych."""
        pass

class VaastavData(FetchData):
    def __init__(self, season_list: list[str]):
        # Vaastav zazwyczaj nie ma jeszcze danych dla najnowszego, trwającego sezonu w tym formacie
        self.season_list = [s for s in season_list if s != "2025-26"]
        self.url_manager = {
            "vaastav": "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/refs/heads/master/data/{}/gws/gw{}.csv"
        }
        self.id_manager = {
            "vaastav" : "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/refs/heads/master/data/{}/players_raw.csv"
        }

    def _fetch_gameweeks(self) -> pd.DataFrame:
        frames = []
        for season in self.season_list:
            for gw in range(1, 39):
                try:
                    url = self.url_manager["vaastav"].format(season, gw)
                    df = pd.read_csv(url)
                    if not df.empty:
                        df = df.assign(season=season, gw=gw).dropna(axis=1, how="all")
                        frames.append(df)
                except Exception:
                    continue
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _fetch_ids(self) -> pd.DataFrame:
        id_frames = []
        for season in self.season_list:
            try:
                url_id = self.id_manager["vaastav"].format(season)
                df = pd.read_csv(url_id, usecols=["id", "first_name", "second_name", "code"])
                df["season"] = season
                df["name"] = df["first_name"] + " " + df["second_name"]
                id_frames.append(df.drop(columns=["first_name", "second_name"]))
            except Exception as e:
                print(f"Błąd pobierania ID dla sezonu {season}: {e}")
        return pd.concat(id_frames, ignore_index=True) if id_frames else pd.DataFrame()

    def _merge_and_clean_data(self, gw_df: pd.DataFrame, id_df: pd.DataFrame) -> pd.DataFrame:
        # Usuwamy ewentualne kolumny 'name' z gw_df przed mergem, aby uniknąć name_x, name_y
        if "name" in gw_df.columns:
            gw_df = gw_df.drop(columns=["name"])

        df = pd.merge(
            gw_df, id_df,
            how="inner",
            left_on=["element", "season"],
            right_on=["id", "season"]
        ).drop(columns=["id"])

        if "value" in df.columns:
            df["value"] = df["value"] / 10

        cols_to_drop = ['opponent_team', 'modified', 'mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw', 'mng_underdog_win', 'mng_win']
        df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
        
        # Kluczowe: usuwamy wszelkie zduplikowane nazwy kolumn, które mogły powstać
        df = df.loc[:, ~df.columns.duplicated()]
            
        return df

    def get_data(self) -> pd.DataFrame:
        gw_df = self._fetch_gameweeks()
        id_df = self._fetch_ids()
        if gw_df.empty or id_df.empty:
            return pd.DataFrame()
        merged_df = self._merge_and_clean_data(gw_df, id_df)
        return merged_df

class FCIData(FetchData):
    def __init__(self, season_list: list):
        self.raw_season = season_list[-1] 
        self.formatted_season = re.sub(r'(\d+)-(\d+)', r'\1-20\2', self.raw_season)
        self.url_manager = {
            "fci" : "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/{}/By%20Gameweek/GW{}/{}.csv",
            "fci_playerstats" : "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/{}/playerstats.csv"
        }
        self.target_columns = [
            'name', 'position', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets',
            'creativity', 'element', 'expected_assists',
            'expected_goal_involvements', 'expected_goals',
            'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored',
            'ict_index', 'influence', 'kickoff_time', 'minutes',
            'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards',
            'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score',
            'threat', 'total_points', 'transfers_balance', 'transfers_in',
            'transfers_out', 'value', 'was_home', 'yellow_cards', 'gw', 'code', "team", "season"
        ]

    def _get_fci_base(self):
        try:
            url_stats = self.url_manager["fci_playerstats"].format(self.formatted_season)
            playerstats = pd.read_csv(url_stats, low_memory=False)
            url_teams = url_stats.replace("playerstats.csv", "teams.csv")
            teams_df = pd.read_csv(url_teams, usecols=["id", "code", "name"])
            teams_df.rename(columns={"name" : "team", "id": "team_id", "code": "team_code"}, inplace=True)
            return playerstats, teams_df
        except Exception as e:
            print(f"Exception FCI Base: {e}")
            return pd.DataFrame(), pd.DataFrame()

    def _fetch_gameweeks(self):
        players_frames = []
        matches_frames = []
        for gw in range(1, 39):
            try:
                url_players = self.url_manager["fci"].format(self.formatted_season, gw, "players")
                df_p = pd.read_csv(url_players, low_memory=False, usecols=["player_id", "first_name", "second_name", "position", "player_code", "team_code"])
                players_frames.append(df_p)
                url_matches = self.url_manager["fci"].format(self.formatted_season, gw, "matches")
                df_m = pd.read_csv(url_matches, low_memory=False, usecols=["match_id", "gameweek", "kickoff_time", "home_team", "away_team", "home_score", "away_score"])
                matches_frames.append(df_m)
            except Exception:
                continue
        if players_frames and matches_frames:            
            return pd.concat(players_frames, ignore_index=True).drop_duplicates(subset=["player_id"]), pd.concat(matches_frames, ignore_index=True)
        return pd.DataFrame(), pd.DataFrame()

    def _clean_and_merge_dfs(self):
        players, matches = self._fetch_gameweeks()
        playerstats, teams_df = self._get_fci_base()
        if players.empty or playerstats.empty:
            return pd.DataFrame()

        players["name"] = players["first_name"] + " " + players["second_name"]
        players = players.drop(columns=["first_name", "second_name"])
        players = pd.merge(players, teams_df, on="team_code", how="left")
        df = pd.merge(playerstats, players, how="left", left_on="id", right_on="player_id")
        df_matches = pd.merge(df, matches, how="left", left_on="gw", right_on="gameweek")
        df_matches = df_matches[(df_matches["team_id"] == df_matches["home_team"]) | (df_matches["team_id"] == df_matches["away_team"])].copy()
        df_matches["was_home"] = df_matches["team_id"] == df_matches["home_team"]
        df_matches["opponent_team"] = np.where(df_matches["was_home"], df_matches["away_team"], df_matches["home_team"])
        
        for col in ["transfers_in_event", "transfers_out_event"]:
            if col in df_matches.columns:
                df_matches[col] = df_matches[col].fillna(0)
        
        if "transfers_in_event" in df_matches.columns and "transfers_out_event" in df_matches.columns:
            df_matches["transfers_balance"] = df_matches["transfers_in_event"] - df_matches["transfers_out_event"]
        
        rename_dict = {
            "id": "element", "ep_this": "xP", "match_id": "fixture", "now_cost": "value",
            "selected_by_percent": "selected", "home_score": "team_h_score", "away_score": "team_a_score",
            "player_code": "code", "event_points": "total_points", "transfers_in_event": "transfers_in",
            "transfers_out_event": "transfers_out"
        }
        df_matches.rename(columns=rename_dict, inplace=True)
        df_matches["round"] = df_matches["gw"]
        df_matches["season"] = self.raw_season 

        return df_matches.loc[:, ~df_matches.columns.duplicated()]

    def get_data(self):
        df = self._clean_and_merge_dfs()
        if df.empty:
            return pd.DataFrame()
        return df[[c for c in self.target_columns if c in df.columns]].copy()



In [29]:
class UnderstatData:
    def __init__(self, season_list: list[str]):
        self.season_list = season_list
        self.understat = sd.Understat(leagues="ENG-Premier League", seasons=[s[2:] for s in self.season_list])

    def _get_player_stats(self) -> pd.DataFrame:
        understat = self.understat
        player_stats = understat.read_player_match_stats().reset_index()
        return player_stats

    def _get_team_stats(self) -> pd.DataFrame:
        understat = self.understat
        return understat.read_team_match_stats().reset_index()

    def _merge_undestat(self)-> pd.DataFrame:
        player_stats = self._get_player_stats()
        team_stats = self._get_team_stats()
        return pd.merge(player_stats, team_stats[['season_id', 'game_id', 'away_points', "date",
       'away_expected_points', 'away_goals', 'away_xg', 'away_np_xg',
       'away_np_xg_difference', 'away_ppda', 'away_deep_completions',
       'home_points', 'home_expected_points', 'home_goals', 'home_xg',
       'home_np_xg', 'home_np_xg_difference', 'home_ppda',
       'home_deep_completions']],
                          how="left", on=["game_id", "season_id"])

    def get_data(self) -> pd.DataFrame:
        df = self._merge_undestat()

        if 'was_home' not in df.columns:
            df['was_home'] = df.apply(
                lambda row: str(row['game']).split(' ', 1)[1].startswith(str(row['team'])), axis=1)
        pairs = [
            ('points', 'home_points', 'away_points'),
            ('expected_points', 'home_expected_points', 'away_expected_points'),
            ('goals', 'home_goals', 'away_goals'),
            ('xg', 'home_xg', 'away_xg'),
            ('np_xg', 'home_np_xg', 'away_np_xg'),
            ('np_xg_difference', 'home_np_xg_difference', 'away_np_xg_difference'),
            ('ppda', 'home_ppda', 'away_ppda'),
            ('deep_completions', 'home_deep_completions', 'away_deep_completions')
        ]

        for base, h_col, a_col in pairs:
            if h_col in df.columns and a_col in df.columns:
                df[f'team_{base}'] = np.where(df['was_home'], df[h_col], df[a_col])
                df[f'opp_{base}'] = np.where(df['was_home'], df[a_col], df[h_col])

                df.drop(columns=[h_col, a_col], inplace=True)

        if 'team_np_xg' in df.columns and 'opp_np_xg' in df.columns:
            df['team_match_np_xg_diff'] = df['team_np_xg'] - df['opp_np_xg']

        if 'team_ppda' in df.columns and 'opp_ppda' in df.columns:
            df['ppda_diff'] = df['team_ppda'] - df['opp_ppda']

        return df


class DataIntegrator:
    def __init__(self):
        self.bridge_url = "https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/refs/heads/main/Understat.csv"

    def _get_bridge(self) -> pd.DataFrame:
        try:
            bridge_df = pd.read_csv(self.bridge_url, usecols=["understat", "code"])
            bridge_df = bridge_df.dropna(subset=['understat', 'code'])
            return bridge_df
        except Exception as e:
            print(f"Błąd podczas pobierania mostu: {e}")
            return pd.DataFrame()

    def integrate(self, fpl_data: pd.DataFrame, understat_data: pd.DataFrame) -> pd.DataFrame:
        bridge = self._get_bridge()
        if bridge.empty: 
            return pd.DataFrame()
        

        understat_with_bridge = pd.merge(
            understat_data[['player_id', 'shots', 'xg_chain', 'xg_buildup', 'key_passes', "date", "season", "team",
                            'was_home', 'team_points', 'opp_points', 'team_expected_points',
       'opp_expected_points', 'team_goals', 'opp_goals', 'team_xg', 'opp_xg',
       'team_np_xg', 'opp_np_xg', 'team_np_xg_difference',
       'opp_np_xg_difference', 'team_ppda', 'opp_ppda',
       'team_deep_completions', 'opp_deep_completions',
       'team_match_np_xg_diff', 'ppda_diff']],
            bridge[['understat', 'code']],
            left_on='player_id',
            right_on='understat',
            how='left'
        )

        df = pd.merge(
            fpl_data,
            understat_with_bridge,
            left_on=['code', 'kickoff_time', "season", "team"],
            right_on=['code', 'date', "season", "team"],
            how='left',
            suffixes=('', '_understat')
        )

        return df



In [2]:
import pandas as pd
pd.set_option("display.max_columns", None)

import numpy as np
import soccerdata as sd
from abc import ABC, abstractmethod
from functools import wraps
import time


seasons = ["2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]

def timeit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"Func: {func.__name__} |: {time.time() - start:3f}")
        return result
    return wrapper

class FetchData(ABC):
    def __init__(self, season_list: list):
        self.season_list = season_list
        self.url_manager = {}

    @abstractmethod
    def get_data(self) -> pd.DataFrame:
        """Główny punkt wejścia do pobierania danych."""
        pass
    
class VaastavData(FetchData):
    def __init__(self, season_list: list[str]):
        self.season_list = season_list
        self.url_manager = {
            "vaastav": "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/refs/heads/master/data/{}/gws/gw{}.csv"
        }
        self.id_manager = {
            "vaastav" : "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/refs/heads/master/data/{}/players_raw.csv"
        }


    def _fetch_gameweeks(self) -> pd.DataFrame:
        frames = []
        for season in self.season_list[:-1]:
            for gw in range(1, 39):
                try:
                    url = self.url_manager["vaastav"].format(season, gw)
                    df = pd.read_csv(url)
                    if not df.empty:
                        df = df.assign(season=season, gw=gw).dropna(axis=1, how="all")
                        frames.append(df)
                except Exception as e:
                    print(f"Błąd pobierania GW {gw} dla sezonu {season}: {e}")
        
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _fetch_ids(self) -> pd.DataFrame:
        id_frames = []
        for season in self.season_list[:-1]:
            try:
                url_id = self.id_manager["vaastav"].format(season)
                df = pd.read_csv(url_id, usecols=["id", "first_name", "second_name", "code"])
                
                df["season"] = season
                df["name"] = df["first_name"] + " " + df["second_name"]
                id_frames.append(df.drop(columns=["first_name", "second_name"]))
            except Exception as e:
                print(f"Błąd pobierania ID dla sezonu {season}: {e}")
        
        return pd.concat(id_frames, ignore_index=True) if id_frames else pd.DataFrame()

    def _merge_and_clean_data(self, gw_df: pd.DataFrame, id_df: pd.DataFrame) -> pd.DataFrame:
        df = pd.merge(
            gw_df, id_df,
            how="inner",
            left_on=["element", "season"],
            right_on=["id", "season"]
        ).drop(columns=["id"])

        if "name_y" in df.columns:
            df.drop(columns=["name_y"], inplace=True)
        if "name_x" in df.columns:
            df.rename(columns={"name_x": "name"}, inplace=True)

        if "value" in df.columns:
            df["value"] = df["value"] / 10

        df = df.drop(columns=['opponent_team', 'modified', 'mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw', 'mng_underdog_win', 'mng_win'])
            
        return df
    
    
    def get_data(self) -> pd.DataFrame:
        gw_df = self._fetch_gameweeks()
        id_df = self._fetch_ids()

        if gw_df.empty or id_df.empty:
            return pd.DataFrame()

        merged_df = self._merge_and_clean_data(gw_df, id_df)
        return merged_df.convert_dtypes()


class FCIData(FetchData):
    def __init__(self, season_list):
        self.season_list = season_list[-1]

        self.url_manager = {
            "fci" : "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2025-2026/By%20Gameweek/GW{}/{}.csv",
            "fci_playerstats" : "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2025-2026/playerstats.csv"
        }
        self.target_columns = [
            'name', 'position', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets',
            'creativity', 'element', 'expected_assists',
            'expected_goal_involvements', 'expected_goals',
            'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored',
            'ict_index', 'influence', 'kickoff_time', 'minutes',
            'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards',
            'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score',
            'threat', 'event_points', 'transfers_balance_event', 'transfers_in_event',
            'transfers_out_event', 'value', 'was_home', 'yellow_cards', 'gw', 'code', "team"
        ]
        
        self.files_manager = ["player_gameweek_stats", "players", "matches", "teams"]
        
    def _get_fci_base(self):
        try:
            playerstats = pd.read_csv(self.url_manager["fci_playerstats"], low_memory=False)
            teams_df = pd.read_csv(self.url_manager["fci_playerstats"].replace("playerstats.csv", "teams.csv"), usecols=["id", "code", "name"])
            teams_df.rename(columns={
                "name" : "team"
            }, inplace=True)
            teams_df.rename(columns={"id": "team_id", "code": "team_code"}, inplace=True)
        
            return playerstats, teams_df
        
        except Exception as e:
            print(f"Exception: {e}")
        
            return pd.DataFrame(), pd.DataFrame()
            
    
    def _fetch_gameweeks(self):
        players_frames = []
        matches_frames = []

        for gw in range(1, 39):
            try:
                url_players = self.url_manager["fci"].format(gw, "players")
                df_p = pd.read_csv(url_players, low_memory=False, usecols=["player_id", "first_name", "second_name", "position", "player_code", "team_code"])
                players_frames.append(df_p)

                
                url_matches = self.url_manager["fci"].format(gw, "matches")
                df_m = pd.read_csv(url_matches, low_memory=False, usecols=["match_id", "gameweek", "kickoff_time", "home_team", "away_team", "home_score", "away_score"])
                matches_frames.append(df_m)
                
                
            except Exception as e:
                print(f"Exception {e}")
                continue
            
        if players_frames and  matches_frames:            
            return pd.concat(players_frames, ignore_index=True).drop_duplicates(subset=["player_id"]), pd.concat(matches_frames, ignore_index=True)

        else:
            return pd.DataFrame(), pd.DataFrame()


    def _clean_and_merge_dfs(self):
        players, matches = self._fetch_gameweeks()
        playerstats, teams_df = self._get_fci_base()
        
        
        players["name"] = players["first_name"] + " " + players["second_name"]
        players = players.drop(columns=["first_name", "second_name"])

        players = pd.merge(players, teams_df, on="team_code", how="left")

        df = pd.merge(playerstats, players, how="left", left_on="id", right_on="player_id")
        
        df_matches = pd.merge(df, matches, how="left", left_on="gw", right_on="gameweek")
        df_matches = df_matches[(df_matches["team_id"] == df_matches["home_team"]) | (df_matches["team_id"] == df_matches["away_team"])].copy()
        
        df_matches["was_home"] = df_matches["team_id"] == df_matches["home_team"]
        df_matches["opponent_team"] = np.where(df_matches["was_home"], df_matches["away_team"], df_matches["home_team"])
        
        for col in ["transfers_in_event", "transfers_out_event"]:
            df_matches[col] = df_matches[col].fillna(0)
        
        df_matches["transfers_balance_event"] = df_matches["transfers_in_event"] - df_matches["transfers_out_event"]

        return df_matches

    
    def _df_prepare(self):
        df = self._clean_and_merge_dfs()
        rename_dict = {
            "id": "element",
            "ep_this": "xP",
            "match_id": "fixture",
            "now_cost": "value",
            "selected_by_percent": "selected",
            "home_score": "team_h_score",
            "away_score": "team_a_score",
            "player_code": "code",
        }
        df.rename(columns=rename_dict, inplace=True)
        df["round"] = df["gw"]
        
        try:
            df = df[self.target_columns].copy()
        except Exception as e:
            print(f"Exception: {e}")
        df.rename(columns={
            "transfers_out_event" : "transfers_out",
            "transfers_balance_event" : "transfers_balance",
            "transfers_in_event" : "transfers_in",
            "event_points" : "total_points"
        }, inplace=True)

        return df.assign(season=self.season_list)

    def get_data(self):
        return self._df_prepare()

class FetchFPL(FetchData):
    def __init__(self, season_list):
        super().__init__(season_list)
        self.team_map = {
            'Southampton': 'Southampton', 'Bournemouth': 'Bournemouth', 'Chelsea': 'Chelsea',
            'Newcastle': 'Newcastle United', 'Leicester': 'Leicester', 'Nott\'m Forest': 'Nottingham Forest',
            'Crystal Palace': 'Crystal Palace', 'Wolves': 'Wolverhampton Wanderers', 'Brentford': 'Brentford',
            'Spurs': 'Tottenham', 'West Ham': 'West Ham', 'Liverpool': 'Liverpool', 'Leeds': 'Leeds',
            'Fulham': 'Fulham', 'Brighton': 'Brighton', 'Man City': 'Manchester City',
            'Man Utd': 'Manchester United', 'Everton': 'Everton', 'Arsenal': 'Arsenal',
            'Aston Villa': 'Aston Villa', 'Sheffield Utd': 'Sheffield United', 'Burnley': 'Burnley',
            'Luton': 'Luton', 'Ipswich': 'Ipswich', 'Sunderland': 'Sunderland',
            'Norwich': 'Norwich', 'Watford': 'Watford'
        }
        self.vaastav = VaastavData(season_list=self.season_list)
        self.fci = FCIData(season_list=self.season_list)
    
    @timeit
    def get_data(self):
        v_df = self.vaastav.get_data()
        f_df = self.fci.get_data()
        
        # Zapewnienie, że obie ramki mają unikalne kolumny przed concat
        v_df = v_df.loc[:, ~v_df.columns.duplicated()]
        f_df = f_df.loc[:, ~f_df.columns.duplicated()]

        df = pd.concat([v_df, f_df], axis=0, ignore_index=True)

        if not df.empty:
            if 'kickoff_time' in df.columns:
                df['kickoff_time'] = (
                    pd.to_datetime(df['kickoff_time'], errors='coerce', utc=True)
                    .dt.tz_localize(None)
                    .astype('datetime64[us]')
                )
                df['date'] = df['kickoff_time'].dt.date
            
            if "fixture" in df.columns:
                df = df.drop(columns=["fixture"])
            
            if "season" in df.columns:
                df["season"] = df["season"].astype(str).str.replace(r"^20", "", regex=True).str.replace("-", "")
            
            if "team" in df.columns:
                df["team"] = df["team"].replace(self.team_map)
            
        return df




class UnderstatData(FetchData):
    def __init__(self, season_list: list[str]):
        self.season_list = season_list
        self.understat = sd.Understat(leagues="ENG-Premier League", seasons=[s[2:] for s in self.season_list])

    def _get_player_stats(self) -> pd.DataFrame:
        understat = self.understat
        player_stats = understat.read_player_match_stats().reset_index()
        return player_stats

    def _get_team_stats(self) -> pd.DataFrame:
        understat = self.understat
        return understat.read_team_match_stats().reset_index()

    def _merge_undestat(self)-> pd.DataFrame:
        player_stats = self._get_player_stats()
        team_stats = self._get_team_stats()
        
        return pd.merge(player_stats, team_stats[['season_id', 'game_id', 'away_points', "date",
       'away_expected_points', 'away_goals', 'away_xg', 'away_np_xg',
       'away_np_xg_difference', 'away_ppda', 'away_deep_completions',
       'home_points', 'home_expected_points', 'home_goals', 'home_xg',
       'home_np_xg', 'home_np_xg_difference', 'home_ppda',
       'home_deep_completions']],
                          how="left", on=["game_id", "season_id"])

    def get_data(self) -> pd.DataFrame:
        df = self._merge_undestat()

        if 'was_home' not in df.columns:
            df['was_home'] = df.apply(
                lambda row: str(row['game']).split(' ', 1)[1].startswith(str(row['team'])), axis=1)
        pairs = [
            ('points', 'home_points', 'away_points'),
            ('expected_points', 'home_expected_points', 'away_expected_points'),
            ('goals', 'home_goals', 'away_goals'),
            ('xg', 'home_xg', 'away_xg'),
            ('np_xg', 'home_np_xg', 'away_np_xg'),
            ('np_xg_difference', 'home_np_xg_difference', 'away_np_xg_difference'),
            ('ppda', 'home_ppda', 'away_ppda'),
            ('deep_completions', 'home_deep_completions', 'away_deep_completions')
        ]

        for base, h_col, a_col in pairs:
            if h_col in df.columns and a_col in df.columns:
                df[f'team_{base}'] = np.where(df['was_home'], df[h_col], df[a_col])
                df[f'opp_{base}'] = np.where(df['was_home'], df[a_col], df[h_col])

                df.drop(columns=[h_col, a_col], inplace=True)

        if 'team_np_xg' in df.columns and 'opp_np_xg' in df.columns:
            df['team_match_np_xg_diff'] = df['team_np_xg'] - df['opp_np_xg']

        if 'team_ppda' in df.columns and 'opp_ppda' in df.columns:
            df['ppda_diff'] = df['team_ppda'] - df['opp_ppda']

        df["date"] = df["date"].dt.date
        return df


class DataIntegrator(FetchData):
    def __init__(self, season_list):
        self.season_list = season_list
        self.fpl_df = FetchFPL(season_list=seasons)
        self.understat = UnderstatData(season_list=self.season_list)

        self.bridge_url = "https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/refs/heads/main/Understat.csv"
    
    
    def _get_bridge(self) -> pd.DataFrame:
        try:
            bridge = pd.read_csv(self.bridge_url, usecols=["understat", "code"])
            bridge = bridge.dropna(subset=['understat', 'code'])
            return bridge

        except Exception as e:
            print(f"Exception: {e}")
            return pd.DataFrame()


    def integrate(self) -> pd.DataFrame:
        bridge = self._get_bridge()
        if bridge.empty:
            return pd.DataFrame()
        fpl_data = self.fpl_df.get_data()
        understat_data = self.understat.get_data()

        understat_with_bridge = pd.merge(
        understat_data[['player_id', 'shots', 'xg_chain', 'xg_buildup', 'key_passes', "date",
                                'was_home', 'team_points', 'opp_points', 'team_expected_points',
                                'opp_expected_points', 'team_goals', 'opp_goals', 'team_xg', 'opp_xg',
                                'team_np_xg', 'opp_np_xg', 'team_np_xg_difference', "team", "season_id",
                                'opp_np_xg_difference', 'team_ppda', 'opp_ppda','team_deep_completions', 
                                'opp_deep_completions', 'team_match_np_xg_diff', 'ppda_diff']],

                bridge[['understat', 'code']],
                left_on='player_id',
                right_on='understat',
                how='left'
            )
        understat_with_bridge.rename(columns={
            "season_id" : "season"
        }, inplace=True)
        understat_with_bridge.season = understat_with_bridge.season.astype(str)
        
        df = pd.merge(
                fpl_data,
                understat_with_bridge,
                left_on=['season', 'code'],
                right_on=['season', 'code'],
                how='left',
                suffixes=('', '_understat')
            )

        return df


    @timeit
    def get_data(self) -> pd.DataFrame:


        return self.integrate()

[05/15/26 06:10:01] INFO     No custom team name replacements found. You can configure these in       ]8;id=1120810;file:///opt/python/lib/python3.13/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=1120811;file:///opt/python/lib/python3.13/site-packages/soccerdata/_config.py#92\92]8;;\
                             /home/onyxia/soccerdata/config/teamname_replacements.json.                            

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=1120817;file:///opt/python/lib/python3.13/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=1120818;file:///opt/python/lib/python3.13/site-packages/soccerdata/_config.py#190\190]8;;\
                             /home/onyxia/soccerdata/config/league_dict.json.                                      

In [31]:
fci = FCIData(season_list=seasons)
fci_data = fci.get_data()

In [32]:
fci_data.shape

(19571, 42)

In [51]:
fci_data.isna().sum()[fci_data.isna().sum()>0]

kickoff_time    2675
team_a_score     163
team_h_score     163
dtype: int64

In [33]:
fpl = FetchFPL(season_list=seasons)
fpl_data = fpl.get_data()

Func: get_data |: 30.100613


In [3]:
fpl = FetchFPL(season_list=seasons)
fpl_data = fpl.get_data()


Func: get_data |: 47.491166


In [4]:
fpl_data.head()

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,goals_conceded,goals_scored,ict_index,influence,kickoff_time,minutes,own_goals,penalties_missed,penalties_saved,red_cards,round,saves,selected,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,season,gw,expected_assists,expected_goal_involvements,expected_goals,expected_goals_conceded,starts,code,date
0,Eric Bailly,DEF,Manchester United,0.0,0,0,0,0,0.0,286,0,0,0.0,0.0,2021-08-14 11:30:00,0,0,0,0,0,1,0,9363.0,1.0,5.0,0.0,0,0,0,0,5.0,True,0,2122,1,<NA>,<NA>,<NA>,<NA>,<NA>,197365,2021-08-14
1,Keinan Davis,FWD,Aston Villa,0.4,0,0,0,0,0.0,49,0,0,0.0,0.0,2021-08-14 14:00:00,0,0,0,0,0,1,0,169789.0,2.0,3.0,0.0,0,0,0,0,4.5,False,0,2122,1,<NA>,<NA>,<NA>,<NA>,<NA>,221239,2021-08-14
2,Ayotomiwa Dele-Bashiru,MID,Watford,0.0,0,0,0,0,0.0,394,0,0,0.0,0.0,2021-08-14 14:00:00,0,0,0,0,0,1,0,4092.0,2.0,3.0,0.0,0,0,0,0,4.5,True,0,2122,1,<NA>,<NA>,<NA>,<NA>,<NA>,175353,2021-08-14
3,James Ward-Prowse,MID,Southampton,2.3,0,0,20,0,30.5,341,3,0,5.2,21.6,2021-08-14 14:00:00,90,0,0,0,0,1,0,299682.0,1.0,3.0,0.0,2,0,0,0,6.5,False,0,2122,1,<NA>,<NA>,<NA>,<NA>,<NA>,101178,2021-08-14
4,Bruno Miguel Borges Fernandes,MID,Manchester United,4.4,0,3,61,0,35.9,277,1,3,20.1,106.2,2021-08-14 11:30:00,90,0,0,0,0,1,0,3381004.0,1.0,5.0,59.0,20,0,0,0,12.0,True,0,2122,1,<NA>,<NA>,<NA>,<NA>,<NA>,141746,2021-08-14


In [6]:
fpl_data.season.unique()

<ArrowStringArray>
['2122', '2223', '2324', '2425', '2526']
Length: 5, dtype: str

In [7]:
fpl_data[fpl_data.season=="2425"].isna().mean()[fpl_data[fpl_data.season=="2425"].isna().mean()>0]

Series([], dtype: float64)

In [8]:
fpl_data[fpl_data.season=="2526"].isna().mean()[fpl_data[fpl_data.season=="2526"].isna().mean()>0]

kickoff_time    1.000000
team_a_score    0.008329
team_h_score    0.008329
date            1.000000
dtype: float64

In [44]:
fpl_data.isna().mean()[fpl_data.isna().mean()>0]


kickoff_time                  0.151886
team_a_score                  0.001265
team_h_score                  0.001265
expected_assists              0.197489
expected_goal_involvements    0.197489
expected_goals                0.197489
expected_goals_conceded       0.197489
starts                        0.197489
date                          0.151886
dtype: float64

In [45]:
bridge = pd.read_csv("https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/refs/heads/main/Understat.csv")
bridge["name"] = bridge["first_name"] + " " +bridge["second_name"]
bridge = bridge.drop(columns=["first_name", "second_name", "web_name"])
bridge.dropna(subset=["understat"], inplace=True)
bridge.understat = bridge.understat.astype(int)

In [46]:
bridge.isna().sum()

code         0
understat    0
name         0
dtype: int64

In [47]:
bridge.head()

,code,understat,name
0,1243,4390,Robert Green
1,1616,277,Alexander Manninger
2,1632,590,Gareth Barry
3,1718,917,John Terry
4,1801,1667,Paul Robinson


In [48]:
understat = UnderstatData(seasons)
qqq = understat.get_data()

[05/13/26 09:46:36] INFO     Saving cached data to /home/onyxia/soccerdata/data/Understat            ]8;id=8037289;file:///opt/python/lib/python3.13/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8037290;file:///opt/python/lib/python3.13/site-packages/soccerdata/_common.py#250\250]8;;\

In [1]:
"""
data.to_parquet(config.TIDY_DATA_PATH,
                engine='pyarrow',
                index=False)"""

"\ndata.to_parquet(config.TIDY_DATA_PATH,\n                engine='pyarrow',\n                index=False)"

In [50]:
import pandas as pd
import numpy as np
import soccerdata as sd
from abc import ABC, abstractmethod
from functools import wraps
import time

# Ustawienia wyświetlania
pd.set_option("display.max_columns", None)

seasons = ["2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]

def timeit(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"Func: {func.__name__} | Czas: {time.time() - start:.3f}s")
        return result
    return wrapper

class FetchData(ABC):
    def __init__(self, season_list: list):
        self.season_list = season_list

    @abstractmethod
    def get_data(self) -> pd.DataFrame:
        pass

class VaastavData(FetchData):
    def __init__(self, season_list: list[str]):
        super().__init__(season_list)
        self.url_base = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/{}/"

    def _fetch_gameweeks(self) -> pd.DataFrame:
        frames = []
        # Pobieramy sezony archiwalne (wszystkie prócz ostatniego, który bierzemy z FCI)
        for season in self.season_list[:-1]:
            for gw in range(1, 39):
                try:
                    url = f"{self.url_base.format(season)}gws/gw{gw}.csv"
                    df = pd.read_csv(url)
                    if not df.empty:
                        df = df.assign(season=season, gw=gw)
                        frames.append(df)
                except:
                    continue 
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _fetch_ids(self) -> pd.DataFrame:
        id_frames = []
        for season in self.season_list[:-1]:
            try:
                url = f"{self.url_base.format(season)}players_raw.csv"
                df = pd.read_csv(url, usecols=["id", "first_name", "second_name", "code"])
                df["season"] = season
                df["name"] = df["first_name"] + " " + df["second_name"]
                id_frames.append(df.drop(columns=["first_name", "second_name"]))
            except:
                continue
        return pd.concat(id_frames, ignore_index=True) if id_frames else pd.DataFrame()

    def get_data(self) -> pd.DataFrame:
        gw_df = self._fetch_gameweeks()
        id_df = self._fetch_ids()
        if gw_df.empty: return pd.DataFrame()

        df = pd.merge(gw_df, id_df, how="inner", left_on=["element", "season"], right_on=["id", "season"])
        df.drop(columns=["id"], errors="ignore", inplace=True)
        
        if "value" in df.columns:
            df["value"] = df["value"] / 10
            
        return df

class FCIData(FetchData):
    def __init__(self, season_list):
        super().__init__(season_list)
        self.current_season = season_list[-1]
        self.base_url = "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/main/data/2025-2026/"
        
    def get_data(self) -> pd.DataFrame:
        try:
            # Główne dane zawodników
            playerstats = pd.read_csv(f"{self.base_url}playerstats.csv", low_memory=False)
            teams = pd.read_csv(f"{self.base_url}teams.csv", usecols=["id", "code", "name"])
            teams.columns = ["team_id", "team_code", "team_name"]
            
            # Pobieranie meczów, aby wyciągnąć kickoff_time
            matches_list = []
            # Zamiast range 38, próbujemy pobrać plik zbiorczy jeśli istnieje, 
            # lub iterujemy bezpiecznie (FPL Core Insights często ma strukturę By Gameweek)
            for gw in range(1, 39):
                try:
                    m_url = f"{self.base_url}By%20Gameweek/GW{gw}/matches.csv"
                    m_df = pd.read_csv(m_url, usecols=["match_id", "gameweek", "kickoff_time", "home_team", "away_team", "home_score", "away_score"])
                    matches_list.append(m_df)
                except:
                    break # Przerwij, jeśli doszliśmy do przyszłych kolejek

            if not matches_list:
                return pd.DataFrame()

            all_matches = pd.concat(matches_list, ignore_index=True)
            
            # Łączenie playerstats z informacją o drużynie
            # W playerstats 'team' to zazwyczaj ID drużyny
            df = pd.merge(playerstats, teams, left_on="team", right_on="team_id", how="left")
            
            # Łączenie z meczami
            df = pd.merge(df, all_matches, left_on=["gw", "team_id"], 
                          right_on=["gameweek", "home_team"], how="left")
            
            # Obsługa meczów wyjazdowych (jeśli nie było w home_team, sprawdź away_team)
            away_mask = df["kickoff_time"].isna()
            if away_mask.any():
                df_away = pd.merge(df[away_mask].drop(columns=all_matches.columns), 
                                   all_matches, left_on=["gw", "team_id"], 
                                   right_on=["gameweek", "away_team"], how="left")
                df = pd.concat([df[~away_mask], df_away], ignore_index=True)

            # Mapowanie nazw kolumn do standardu Vaastav
            rename_dict = {
                "id": "element",
                "ep_this": "xP",
                "match_id": "fixture",
                "now_cost": "value",
                "selected_by_percent": "selected",
                "home_score": "team_h_score",
                "away_score": "team_a_score",
                "event_points": "total_points",
                "team_name": "team"
            }
            df.rename(columns=rename_dict, inplace=True)
            df["season"] = self.current_season
            df["was_home"] = df["team_id"] == df["home_team"]
            
            return df
        except Exception as e:
            print(f"FCI Error: {e}")
            return pd.DataFrame()

class FetchFPL(FetchData):
    def __init__(self, season_list):
        super().__init__(season_list)
        self.team_map = {
            'Man City': 'Manchester City', 'Man Utd': 'Manchester United',
            'Spurs': 'Tottenham', 'Wolves': 'Wolverhampton Wanderers',
            'Nott\'m Forest': 'Nottingham Forest', 'Sheffield Utd': 'Sheffield United'
        }

    @timeit
    def get_data(self):
        v_df = VaastavData(self.season_list).get_data()
        f_df = FCIData(self.season_list).get_data()

        df = pd.concat([v_df, f_df], axis=0, ignore_index=True)
        
        if df.empty: return df

        # Fix daty
        if 'kickoff_time' in df.columns:
            df['kickoff_time'] = pd.to_datetime(df['kickoff_time'], errors='coerce')
            df['date'] = df['kickoff_time'].dt.date

        # Normalizacja sezonu (np. 2021-22 -> 2122)
        if "season" in df.columns:
            df["season"] = df["season"].astype(str).str.replace("20", "", regex=False).str.replace("-", "", regex=False)
        
        # Mapowanie nazw drużyn
        if "team" in df.columns:
            df["team"] = df["team"].replace(self.team_map)
            
        return df

class UnderstatData(FetchData):
    def __init__(self, season_list: list[str]):
        super().__init__(season_list)
        # Soccerdata używa formatu "21", "22" itd.
        sd_seasons = [s[2:4] + s[5:7] for s in self.season_list]
        self.understat = sd.Understat(leagues="ENG-Premier League", seasons=sd_seasons)

    def get_data(self) -> pd.DataFrame:
        try:
            p_stats = self.understat.read_player_match_stats().reset_index()
            t_stats = self.understat.read_team_match_stats().reset_index()
            
            # Wstępne czyszczenie sezonu w Understat do formatu 2122
            p_stats['season'] = p_stats['season'].astype(str).str.replace("-", "", regex=False)
            
            df = pd.merge(p_stats, t_stats, on=["game_id", "season", "date"], suffixes=("", "_team"))
            df["date"] = pd.to_datetime(df["date"]).dt.date
            return df
        except Exception as e:
            print(f"Understat Error: {e}")
            return pd.DataFrame()

class DataIntegrator:
    def __init__(self, season_list):
        self.season_list = season_list
        self.fpl_loader = FetchFPL(season_list)
        self.understat_loader = UnderstatData(season_list)
        self.bridge_url = "https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/main/Understat.csv"

    @timeit
    def get_data(self):
        fpl_df = self.fpl_loader.get_data()
        und_df = self.understat_loader.get_data()
        bridge = pd.read_csv(self.bridge_url, usecols=["understat", "code"])
        
        # Łączymy Understat z mostkiem ID
        und_df = pd.merge(und_df, bridge, left_on="player_id", right_on="understat", how="left")
        
        # Finalne połączenie
        # Kluczowe: upewnienie się, że 'code' i 'season' są tego samego typu
        fpl_df['code'] = fpl_df['code'].astype(float)
        und_df['code'] = und_df['code'].astype(float)
        fpl_df['season'] = fpl_df['season'].astype(str)
        und_df['season'] = und_df['season'].astype(str)
        
        final_df = pd.merge(
            fpl_df, 
            und_df.drop(columns=['team', 'date'], errors='ignore'), 
            on=['season', 'code'], 
            how='left', 
            suffixes=('', '_und')
        )
        return final_df

# Uruchomienie
integrator = DataIntegrator(seasons)
full_data = integrator.get_data()

print("\nSprawdzenie braków dla 2526:")
print(full_data[full_data.season=="2526"][['kickoff_time', 'date', 'team_h_score']].isna().mean())


[05/13/26 09:46:39] INFO     Saving cached data to /home/onyxia/soccerdata/data/Understat            ]8;id=8037295;file:///opt/python/lib/python3.13/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8037296;file:///opt/python/lib/python3.13/site-packages/soccerdata/_common.py#250\250]8;;\

FCI Error: 'team'
Func: get_data | Czas: 35.270s
Understat Error: 'date'


KeyError: 'player_id'